# Quantitative Trading Tool - Quick Start

This notebook demonstrates the basic usage of the quantitative trading tool.

In [ ]:
# Setup path
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Data Fetching

In [ ]:
from core import DataManager

dm = DataManager()

# Fetch US stock data
aapl = dm.fetch_ohlcv('AAPL', '2023-01-01', '2023-12-31')
print(f"AAPL: {len(aapl)} records")
aapl.head()

In [ ]:
# Plot candlestick chart
fig = go.Figure(data=[go.Candlestick(
    x=aapl.index,
    open=aapl['open'],
    high=aapl['high'],
    low=aapl['low'],
    close=aapl['close']
)])
fig.update_layout(title='AAPL Stock Price', xaxis_title='Date', yaxis_title='Price')
fig.show()

## 2. Backtesting

In [ ]:
from core import BacktestEngine

# Fetch multi-asset data
symbols = ['AAPL', 'MSFT']
prices = {}
for symbol in symbols:
    df = dm.fetch_ohlcv(symbol, '2022-01-01', '2023-12-31')
    if df is not None:
        prices[symbol] = df['close']

prices_df = pd.DataFrame(prices).dropna()
print(f"Combined data: {len(prices_df)} records")

In [ ]:
# Run mean reversion backtest
engine = BacktestEngine()

result = engine.run_mean_reversion(
    prices_df,
    lookback=20,
    entry_zscore=2.0,
    exit_zscore=0.5,
    initial_capital=100000
)

print(f"Total Return: {result.total_return:.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio:.2f}")
print(f"Max Drawdown: {result.max_drawdown:.2%}")
print(f"Win Rate: {result.win_rate:.2%}")

In [ ]:
# Plot equity curve
if result.equity_curve is not None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=result.equity_curve.index,
        y=result.equity_curve.values,
        name='Equity'
    ))
    fig.update_layout(title='Equity Curve', xaxis_title='Date', yaxis_title='Portfolio Value')
    fig.show()

## 3. Portfolio Optimization

In [ ]:
from core import PortfolioOptimizer

# Fetch more assets
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
prices = {}
for symbol in symbols:
    df = dm.fetch_ohlcv(symbol, '2022-01-01', '2023-12-31')
    if df is not None:
        prices[symbol] = df['close']

prices_df = pd.DataFrame(prices).dropna()
print(f"Assets: {list(prices_df.columns)}")

In [ ]:
# Maximum Sharpe ratio optimization
optimizer = PortfolioOptimizer()

result = optimizer.max_sharpe(prices_df, risk_free_rate=0.02)

print("Optimal Weights:")
for symbol, weight in result.weights.items():
    if weight > 0.01:
        print(f"  {symbol}: {weight:.2%}")

print(f"\nExpected Return: {result.expected_return:.2%}")
print(f"Volatility: {result.volatility:.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio:.2f}")

In [ ]:
# Plot weights
weights = {k: v for k, v in result.weights.items() if v > 0.01}
fig = go.Figure(data=[go.Pie(
    labels=list(weights.keys()),
    values=list(weights.values()),
    hole=0.4
)])
fig.update_layout(title='Optimal Portfolio Allocation')
fig.show()

## 4. Risk Analysis

In [ ]:
from core import RiskCalculator

# Calculate returns
returns = prices_df['AAPL'].pct_change().dropna()

# Create risk calculator
calc = RiskCalculator(returns, risk_free_rate=0.02)

# Calculate all metrics
metrics = calc.calculate_all()
print(metrics.summary())

In [ ]:
# VaR comparison
print("Value at Risk (VaR) Comparison:")
print(f"  Historical VaR (95%): {calc.var_historical(0.95):.2%}")
print(f"  Parametric VaR (95%): {calc.var_parametric(0.95):.2%}")
print(f"  CVaR (95%): {calc.cvar(0.95):.2%}")

In [ ]:
# Plot return distribution
fig = go.Figure(data=[go.Histogram(x=returns, nbinsx=50)])
fig.update_layout(
    title='Return Distribution',
    xaxis_title='Daily Return',
    yaxis_title='Frequency'
)
fig.show()

## 5. Pairs Trading

In [ ]:
from strategies import PairsTradingStrategy

# Test cointegration
strategy = PairsTradingStrategy()

is_coint, hedge_ratio, p_value = strategy.test_cointegration(
    prices_df['AAPL'],
    prices_df['MSFT']
)

print(f"Cointegrated: {is_coint}")
print(f"Hedge Ratio: {hedge_ratio:.4f}")
print(f"P-Value: {p_value:.4f}")

In [ ]:
# Calculate spread and z-score
spread = prices_df['AAPL'] - hedge_ratio * prices_df['MSFT']
zscore = (spread - spread.rolling(20).mean()) / spread.rolling(20).std()

# Plot spread and z-score
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Spread', 'Z-Score'))

fig.add_trace(go.Scatter(x=spread.index, y=spread, name='Spread'), row=1, col=1)
fig.add_trace(go.Scatter(x=zscore.index, y=zscore, name='Z-Score'), row=2, col=1)
fig.add_hline(y=2, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=-2, line_dash='dash', line_color='green', row=2, col=1)

fig.update_layout(height=600, title='AAPL/MSFT Pairs Trading Analysis')
fig.show()

## Next Steps

- Run `streamlit run app.py` for the full web interface
- Check the `examples/` directory for more detailed examples
- Read the `README.md` for complete documentation